# MedicaInsure - Insurance Premium Prediction

## Problem Statement

**MedicaInsure** is a medical insurance provider. Leveraging customer information is of paramount importance for most businesses. In the case of an insurance company, analysis of customer attributes like age, sex, smoking habits, etc. can be crucial in making decisions regarding the premium amount to be charged.

### Objective
Build a regression model to predict insurance premium charges based on customer attributes.

### Dataset Attributes
- **age**: Age of the policyholder
- **sex**: Gender of the policyholder
- **bmi**: Body Mass Index of the policyholder
- **children**: Number of children/dependents of the policyholder
- **smoker**: Whether the policyholder is a smoker or not
- **region**: Region where the policyholder belongs to
- **charges**: Premium charged to the policyholder (Target Variable)

---
## 1. Import Libraries

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Scikit-learn imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Plot settings
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

---
## 2. Load and Explore Data

In [ ]:
# Load the dataset
df = pd.read_csv('insurance.csv')

# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Display last few rows
print("Last 5 rows of the dataset:")
df.tail()

In [ ]:
# Dataset shape
print(f"Dataset Shape: {df.shape}")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

In [ ]:
# Data types and info
print("\nDataset Info:")
df.info()

In [ ]:
# Statistical summary of numerical columns
print("\nStatistical Summary (Numerical Columns):")
df.describe()

In [ ]:
# Statistical summary of categorical columns
print("\nStatistical Summary (Categorical Columns):")
df.describe(include='object')

In [ ]:
# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicate rows
print(f"\nNumber of duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Unique values in categorical columns
categorical_cols = df.select_dtypes(include='object').columns
print("\nUnique values in categorical columns:")
for col in categorical_cols:
    print(f"\n{col}: {df[col].unique()}")
    print(f"Value counts:\n{df[col].value_counts()}")

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Distribution of Target Variable (Charges)

In [ ]:
# Distribution of charges
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['charges'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Charges ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Insurance Charges', fontsize=14)
axes[0].axvline(df['charges'].mean(), color='red', linestyle='--', label=f'Mean: ${df["charges"].mean():,.2f}')
axes[0].axvline(df['charges'].median(), color='green', linestyle='--', label=f'Median: ${df["charges"].median():,.2f}')
axes[0].legend()

# Box plot
sns.boxplot(y=df['charges'], ax=axes[1], color='steelblue')
axes[1].set_ylabel('Charges ($)', fontsize=12)
axes[1].set_title('Box Plot of Insurance Charges', fontsize=14)

plt.tight_layout()
plt.show()

print(f"\nCharges Statistics:")
print(f"Mean: ${df['charges'].mean():,.2f}")
print(f"Median: ${df['charges'].median():,.2f}")
print(f"Std Dev: ${df['charges'].std():,.2f}")
print(f"Min: ${df['charges'].min():,.2f}")
print(f"Max: ${df['charges'].max():,.2f}")

### 3.2 Distribution of Numerical Features

In [ ]:
# Distribution of numerical features
numerical_cols = ['age', 'bmi', 'children']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=20, edgecolor='black', alpha=0.7, color='coral')
    axes[i].set_xlabel(col.capitalize(), fontsize=12)
    axes[i].set_ylabel('Frequency', fontsize=12)
    axes[i].set_title(f'Distribution of {col.capitalize()}', fontsize=14)

plt.tight_layout()
plt.show()

### 3.3 Categorical Features Analysis

In [ ]:
# Distribution of categorical features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

categorical_features = ['sex', 'smoker', 'region']
colors = ['lightblue', 'lightcoral', 'lightgreen']

for i, col in enumerate(categorical_features):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color=colors[i], edgecolor='black')
    axes[i].set_xlabel(col.capitalize(), fontsize=12)
    axes[i].set_ylabel('Count', fontsize=12)
    axes[i].set_title(f'Distribution of {col.capitalize()}', fontsize=14)
    axes[i].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 3.4 Relationship Between Features and Charges

In [ ]:
# Age vs Charges
plt.figure(figsize=(10, 6))
plt.scatter(df['age'], df['charges'], alpha=0.6, c=df['smoker'].map({'yes': 'red', 'no': 'blue'}), edgecolors='black', linewidths=0.5)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.title('Age vs Insurance Charges (Colored by Smoker Status)', fontsize=14)
# Create legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='red', label='Smoker'),
                   Patch(facecolor='blue', label='Non-Smoker')]
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

In [ ]:
# BMI vs Charges
plt.figure(figsize=(10, 6))
plt.scatter(df['bmi'], df['charges'], alpha=0.6, c=df['smoker'].map({'yes': 'red', 'no': 'blue'}), edgecolors='black', linewidths=0.5)
plt.xlabel('BMI', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.title('BMI vs Insurance Charges (Colored by Smoker Status)', fontsize=14)
legend_elements = [Patch(facecolor='red', label='Smoker'),
                   Patch(facecolor='blue', label='Non-Smoker')]
plt.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

In [ ]:
# Smoker vs Charges
plt.figure(figsize=(8, 6))
sns.boxplot(x='smoker', y='charges', data=df, palette={'no': 'lightblue', 'yes': 'salmon'})
plt.xlabel('Smoker', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.title('Insurance Charges by Smoking Status', fontsize=14)
plt.tight_layout()
plt.show()

print("\nMean charges by smoker status:")
print(df.groupby('smoker')['charges'].mean())

In [ ]:
# Sex vs Charges
plt.figure(figsize=(8, 6))
sns.boxplot(x='sex', y='charges', data=df, palette='Set2')
plt.xlabel('Sex', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.title('Insurance Charges by Gender', fontsize=14)
plt.tight_layout()
plt.show()

print("\nMean charges by gender:")
print(df.groupby('sex')['charges'].mean())

In [ ]:
# Region vs Charges
plt.figure(figsize=(10, 6))
sns.boxplot(x='region', y='charges', data=df, palette='viridis')
plt.xlabel('Region', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.title('Insurance Charges by Region', fontsize=14)
plt.tight_layout()
plt.show()

print("\nMean charges by region:")
print(df.groupby('region')['charges'].mean().sort_values(ascending=False))

In [ ]:
# Number of children vs Charges
plt.figure(figsize=(10, 6))
sns.boxplot(x='children', y='charges', data=df, palette='coolwarm')
plt.xlabel('Number of Children', fontsize=12)
plt.ylabel('Charges ($)', fontsize=12)
plt.title('Insurance Charges by Number of Children', fontsize=14)
plt.tight_layout()
plt.show()

print("\nMean charges by number of children:")
print(df.groupby('children')['charges'].mean())

### 3.5 Correlation Analysis

In [ ]:
# Create a copy for correlation analysis
df_encoded = df.copy()

# Encode categorical variables for correlation
df_encoded['sex'] = df_encoded['sex'].map({'male': 1, 'female': 0})
df_encoded['smoker'] = df_encoded['smoker'].map({'yes': 1, 'no': 0})

# One-hot encode region
region_dummies = pd.get_dummies(df_encoded['region'], prefix='region', drop_first=True)
df_encoded = pd.concat([df_encoded, region_dummies], axis=1)
df_encoded.drop('region', axis=1, inplace=True)

# Correlation matrix
plt.figure(figsize=(12, 8))
correlation_matrix = df_encoded.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f', linewidths=0.5)
plt.title('Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

print("\nCorrelation with Charges:")
print(correlation_matrix['charges'].sort_values(ascending=False))

### 3.6 Key Insights from EDA

1. **Smoker Status**: This is the most significant factor affecting insurance charges. Smokers pay significantly higher premiums.
2. **Age**: Positive correlation with charges - older policyholders tend to have higher premiums.
3. **BMI**: Moderate positive correlation with charges, especially for smokers.
4. **Gender**: Slight difference in average charges between males and females.
5. **Region**: Some regional variation exists, but less significant than smoking and age.
6. **Children**: Number of dependents has some impact on charges.

---
## 4. Data Preprocessing

In [ ]:
# Create a copy for modeling
df_model = df.copy()

# Encode categorical variables
# Binary encoding for sex and smoker
df_model['sex'] = df_model['sex'].map({'male': 1, 'female': 0})
df_model['smoker'] = df_model['smoker'].map({'yes': 1, 'no': 0})

# One-hot encoding for region
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)

print("Preprocessed DataFrame:")
df_model.head()

In [ ]:
# Display columns after preprocessing
print("Columns after preprocessing:")
print(df_model.columns.tolist())

In [ ]:
# Separate features and target
X = df_model.drop('charges', axis=1)
y = df_model['charges']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

In [ ]:
# Display feature names
print("\nFeatures:")
print(X.columns.tolist())

---
## 5. Train-Test Split

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")
print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

---
## 6. Model Building - Linear Regression

In [ ]:
# Initialize and train the Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Linear Regression model trained successfully!")

In [ ]:
# Model coefficients
print("Model Coefficients:")
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print(coefficients)
print(f"\nIntercept: {lr_model.intercept_:.2f}")

In [ ]:
# Visualize coefficients
plt.figure(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in coefficients['Coefficient']]
plt.barh(coefficients['Feature'], coefficients['Coefficient'], color=colors, edgecolor='black')
plt.xlabel('Coefficient Value', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Coefficients in Linear Regression Model', fontsize=14)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

---
## 7. Model Predictions

In [ ]:
# Make predictions on training and testing sets
y_train_pred = lr_model.predict(X_train)
y_test_pred = lr_model.predict(X_test)

print("Predictions generated successfully!")

In [ ]:
# Compare actual vs predicted values
comparison_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_test_pred,
    'Difference': y_test.values - y_test_pred
})

print("Sample predictions (first 10):")
comparison_df.head(10)

---
## 8. Model Evaluation

In [ ]:
# Calculate evaluation metrics
def evaluate_model(y_true, y_pred, dataset_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"\n{dataset_name} Set Metrics:")
    print(f"{'='*40}")
    print(f"Mean Squared Error (MSE): {mse:,.2f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:,.2f}")
    print(f"Mean Absolute Error (MAE): {mae:,.2f}")
    print(f"R-squared (R²): {r2:.4f}")
    print(f"R-squared (%): {r2*100:.2f}%")
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

# Evaluate on training set
train_metrics = evaluate_model(y_train, y_train_pred, "Training")

# Evaluate on testing set
test_metrics = evaluate_model(y_test, y_test_pred, "Testing")

In [ ]:
# Create a summary table
metrics_summary = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'MAE', 'R²'],
    'Training': [train_metrics['MSE'], train_metrics['RMSE'], train_metrics['MAE'], train_metrics['R2']],
    'Testing': [test_metrics['MSE'], test_metrics['RMSE'], test_metrics['MAE'], test_metrics['R2']]
})

print("\nMetrics Summary:")
metrics_summary

### 8.1 Visualize Model Performance

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.5, color='blue', edgecolors='black', linewidths=0.5)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Charges ($)', fontsize=12)
axes[0].set_ylabel('Predicted Charges ($)', fontsize=12)
axes[0].set_title(f'Training Set: Actual vs Predicted\nR² = {train_metrics["R2"]:.4f}', fontsize=14)
axes[0].legend()

# Testing set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, color='green', edgecolors='black', linewidths=0.5)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Charges ($)', fontsize=12)
axes[1].set_ylabel('Predicted Charges ($)', fontsize=12)
axes[1].set_title(f'Testing Set: Actual vs Predicted\nR² = {test_metrics["R2"]:.4f}', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training residuals
train_residuals = y_train - y_train_pred
axes[0].scatter(y_train_pred, train_residuals, alpha=0.5, color='blue', edgecolors='black', linewidths=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Charges ($)', fontsize=12)
axes[0].set_ylabel('Residuals ($)', fontsize=12)
axes[0].set_title('Training Set: Residual Plot', fontsize=14)

# Testing residuals
test_residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, test_residuals, alpha=0.5, color='green', edgecolors='black', linewidths=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Charges ($)', fontsize=12)
axes[1].set_ylabel('Residuals ($)', fontsize=12)
axes[1].set_title('Testing Set: Residual Plot', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training residuals distribution
axes[0].hist(train_residuals, bins=30, edgecolor='black', alpha=0.7, color='blue')
axes[0].set_xlabel('Residuals ($)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Training Set: Distribution of Residuals', fontsize=14)
axes[0].axvline(x=0, color='r', linestyle='--', lw=2)

# Testing residuals distribution
axes[1].hist(test_residuals, bins=30, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('Residuals ($)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Testing Set: Distribution of Residuals', fontsize=14)
axes[1].axvline(x=0, color='r', linestyle='--', lw=2)

plt.tight_layout()
plt.show()

---
## 9. Feature Importance Analysis

In [ ]:
# Feature importance based on absolute coefficient values
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_,
    'Abs_Coefficient': np.abs(lr_model.coef_)
}).sort_values(by='Abs_Coefficient', ascending=False)

print("Feature Importance (by absolute coefficient):")
feature_importance

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Abs_Coefficient'], color='teal', edgecolor='black')
plt.xlabel('Absolute Coefficient Value', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('Feature Importance in Linear Regression Model', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

---
## 10. Conclusions and Recommendations

### Key Findings

1. **Smoking Status** is the most significant predictor of insurance charges. Smokers pay substantially higher premiums.

2. **Age** has a positive relationship with charges - older policyholders tend to have higher insurance costs.

3. **BMI (Body Mass Index)** also contributes to higher charges, particularly for smokers.

4. **Regional Differences** exist but are less significant compared to smoking and age.

5. **Number of Children** has a moderate impact on insurance premiums.

### Model Performance

- The Linear Regression model explains a significant portion of the variance in insurance charges.
- The R² score indicates the model's ability to predict insurance premiums based on customer attributes.
- Residual analysis shows the model's prediction errors are relatively evenly distributed.

### Business Recommendations

1. **Risk-Based Pricing**: Implement tiered pricing based on smoking status as the primary risk factor.

2. **Health Incentives**: Consider offering discounts or incentives for customers who quit smoking or maintain healthy BMI levels.

3. **Age-Adjusted Premiums**: Structure premiums with age-based adjustments reflecting the increased risk with age.

4. **Regional Analysis**: Further investigate regional differences to optimize pricing strategies across different markets.

5. **Model Enhancement**: Consider exploring more advanced models (Ridge, Lasso, Random Forest) to potentially improve prediction accuracy.

In [ ]:
print("\n" + "="*60)
print("CASE STUDY COMPLETED SUCCESSFULLY!")
print("="*60)